# Ablation: removing a probe's direction

Does the Marathi case probe still work once the direction a Hindi probe uses has
been taken out of mBERT's features, and the other way round? Runs at the main
setting (layer 7, first-piece) with the five seeds in `config.yaml`, in both
directions:

- **Hindi → Marathi:** remove directions found by the Hindi probe, train a
  Marathi probe on the projected features, score it on Marathi dev.
- **Marathi → Hindi:** remove directions found by the Marathi probe, train a
  Hindi probe on the projected features, score it on Hindi dev.

The directions are removed one at a time with iterative nullspace projection
(INLP): train a probe on the source language, take its weight vector `w`, project
every feature vector onto the space orthogonal to `w`, train a new probe on the
projected features, and repeat. After k rounds the features have no component
along any of the k directions. The target-language probe is then retrained from
scratch on features projected the same way (it chooses its own C by
cross-validation, as every probe in the project does) and scored on the target
dev split. The curve over k is the result.

Two controls remove the same number of directions k, but chosen at random:

- **Random, isotropic:** k random orthonormal directions in the 768-dimensional
  space.
- **Random, data covariance:** k orthonormal directions drawn in proportion to
  how much the source-language features vary along them. mBERT's features are
  far from isotropic (a few dimensions carry most of the variance), so removing
  an isotropic direction may barely touch the data. This control removes
  directions that carry real variance, which is what an INLP direction does.

If the case information lives in a direction the two languages share, removing
the source language's directions should hurt the target probe more than removing
the same number of random ones. The notebook also tracks the source-language
probe's accuracy after removal, to check that INLP removes what it is meant to,
and how much of the target probe's weight vector lies in the removed subspace.

In [ ]:
import json
import os
import warnings
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml
from joblib import Parallel, delayed
from matplotlib.colors import to_rgb
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupKFold

# scikit-learn's FutureWarnings about LogisticRegressionCV would otherwise print once per probe, in every worker.
warnings.filterwarnings("ignore", category=FutureWarning)
os.environ["PYTHONWARNINGS"] = "ignore::FutureWarning"

ROOT = Path.cwd()
CONFIG = yaml.safe_load((ROOT / "config.yaml").read_text())
CACHE = ROOT / "data" / "cache"
samples = json.loads((CACHE / "samples.json").read_text())
LAYER, POOLING = CONFIG["main_setting"]["layer"], CONFIG["main_setting"]["pooling"]
SEEDS = [str(seed) for seed in CONFIG["seeds"]]

# Numbers of directions removed at which the target probe is retrained and scored.
KS = [0, 1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64]
# Labelled sentences the target probe trains on (the same budgets as the controls in combine.ipynb).
TARGET_SIZES = [25, 100, 800]
# Sentences the source probe, and so the removed directions, are learned from.
SOURCE_SENTENCES = CONFIG["hindi_train_sentences"]
# Random subspaces per seed for each control.
RANDOM_REPEATS = 3
# (source language, target language)
DIRECTIONS = {"hi_to_mr": ("hi", "mr"), "mr_to_hi": ("mr", "hi")}
LANGUAGES = {"hi": "Hindi", "mr": "Marathi"}
# One worker per (direction, seed) job; set to 1 to run in this process.
N_JOBS = min(len(SEEDS) * len(DIRECTIONS), os.cpu_count() or 1)

## Hindi dev features

`data.ipynb` caches features for the Marathi splits and for the sampled Hindi
training sentences, but not for Hindi dev, which the Marathi → Hindi direction
needs. This cell encodes the Hindi dev split the same way `data.ipynb` does
(first- and last-piece vectors of every target noun at all 13 layers, same file
names and layout) and skips itself if the files are already there.

In [ ]:
def read_sentences(language: str, split: str) -> list[dict]:
    """Each sentence's words and its targets as (position, label); same rule as data.ipynb."""
    import conllu

    prefix = CONFIG["treebanks"][language]["prefix"]
    path = ROOT / "data" / "raw" / language / f"{prefix}-ud-{split}.conllu"
    sentences = []
    with path.open(encoding="utf-8") as stream:
        for sentence in conllu.parse_incr(stream):
            words = [token for token in sentence if isinstance(token["id"], int)]
            targets = [
                (i, token["feats"]["Case"]) for i, token in enumerate(words)
                if token["upos"] == "NOUN" and (token["feats"] or {}).get("Case") in CONFIG["labels"]
            ]
            sentences.append({"words": [token["form"] for token in words], "targets": targets})
    return sentences


def cache_features(language: str, split: str, batch_size: int = 32) -> None:
    """Encode every sentence with a target and save its nouns' features, as data.ipynb does."""
    import time

    import torch
    from transformers import AutoModel, AutoTokenizer

    started = time.time()
    tokenizer = AutoTokenizer.from_pretrained(CONFIG["model"])
    device = "xpu" if hasattr(torch, "xpu") and torch.xpu.is_available() else "cpu"
    model = AutoModel.from_pretrained(CONFIG["model"]).to(device).eval()
    piece = {"first": 0, "last": -1}
    sentences = read_sentences(language, split)
    ids = [i for i, s in enumerate(sentences) if s["targets"]]
    nouns = [
        {"sentence": i, "word": w, "form": sentences[i]["words"][w], "label": label}
        for i in ids for w, label in sentences[i]["targets"]
    ]
    shape = (model.config.num_hidden_layers + 1, len(nouns), model.config.hidden_size)
    out = {
        pooling: np.lib.format.open_memmap(
            CACHE / f"{language}_{split}_{pooling}.npy", mode="w+", dtype=np.float32, shape=shape)
        for pooling in CONFIG["poolings"]
    }
    row = 0
    for start in range(0, len(ids), batch_size):
        batch = [sentences[i] for i in ids[start:start + batch_size]]
        encoded = tokenizer([s["words"] for s in batch], is_split_into_words=True,
                            padding=True, return_tensors="pt")
        rows, positions = [], {pooling: [] for pooling in out}
        for b, sentence in enumerate(batch):
            word_ids = encoded.word_ids(b)
            for w, _ in sentence["targets"]:
                pieces = [k for k, word_id in enumerate(word_ids) if word_id == w]
                rows.append(b)
                for pooling in out:
                    positions[pooling].append(pieces[piece[pooling]])
        with torch.inference_mode():
            states = torch.stack(model(**encoded.to(device), output_hidden_states=True).hidden_states)
        for pooling, array in out.items():
            array[:, row:row + len(rows)] = states[:, rows, positions[pooling]].float().cpu().numpy()
        row += len(rows)
    for array in out.values():
        array.flush()
    (CACHE / f"{language}_{split}_nouns.json").write_text(json.dumps(nouns, ensure_ascii=False) + "\n")
    print(f"{language}/{split}: {len(ids)} sentences, {len(nouns)} nouns, {time.time() - started:.0f}s on {device}")


if not all((CACHE / f"hi_dev_{name}").exists() for name in ["first.npy", "last.npy", "nouns.json"]):
    cache_features("hi", "dev")
else:
    print("hi/dev features already cached")

## Data

The same `Split` as in `combine.ipynb`: one split's features at the main
setting, its labels (True = Nom) and its nouns by sentence.

In [ ]:
class Split:
    """One split's cached features at the main setting, its labels (True = Nom) and its nouns by sentence."""

    def __init__(self, language: str, split: str):
        nouns = json.loads((CACHE / f"{language}_{split}_nouns.json").read_text())
        self.x = np.asarray(np.load(CACHE / f"{language}_{split}_{POOLING}.npy", mmap_mode="r")[LAYER])
        self.y = np.array([noun["label"] == "Nom" for noun in nouns])
        self.sentence = np.array([noun["sentence"] for noun in nouns])
        self.by_sentence = defaultdict(list)
        for k, noun in enumerate(nouns):
            self.by_sentence[noun["sentence"]].append(k)

    def rows(self, sentences) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Features, labels and sentence ids of the nouns in these sentences."""
        keep = [k for s in sentences for k in self.by_sentence[s]]
        return self.x[keep], self.y[keep], self.sentence[keep]


train = {"hi": Split("hi", "train"), "mr": Split("mr", "train")}
dev = {"hi": Split("hi", "dev"), "mr": Split("mr", "dev")}
majority = {language: max(split.y.mean(), 1 - split.y.mean()) for language, split in dev.items()}
for language in LANGUAGES:
    print(f"{LANGUAGES[language]} dev: {len(dev[language].y)} nouns, majority {majority[language]:.3f}")

## Removing directions

`inlp` runs the iteration on the source language. Every round trains a probe
with the same C, chosen once by cross-validation on the unprojected features
(re-choosing C every round would make this slow and would let the probe's
strength change from round to round). The probe's accuracy on the projected
source dev split after i removals is kept, so the source curve is available at
every i. A new direction is made orthogonal to the earlier ones before it is
removed, so the removed directions are exactly orthonormal.

`random_basis` gives the controls' directions. The directions are orthonormal and
nested: the first k of them are the k directions removed at k. In the data
covariance control the directions are random combinations of the centred source
features, so they follow the directions in which the source features vary.

In [ ]:
CS = [0.001, 0.01, 0.1, 1.0, 10.0]


def probe(x: np.ndarray, y: np.ndarray, groups: np.ndarray) -> LogisticRegressionCV:
    """Train a probe, choosing C by 5-fold cross-validation that keeps each sentence's nouns together."""
    folds = list(GroupKFold(n_splits=5).split(x, y, groups))
    return LogisticRegressionCV(Cs=CS, cv=folds, scoring="neg_log_loss", max_iter=5000).fit(x, y)


def project(x: np.ndarray, basis: np.ndarray) -> np.ndarray:
    """Remove from every row of x its component in the span of the orthonormal rows of basis."""
    return x if len(basis) == 0 else x - (x @ basis.T) @ basis


def inlp(x, y, groups, x_dev, y_dev, k_max: int):
    """Iterative nullspace projection of a source language's probe.

    Returns the k_max directions (as orthonormal rows), the source dev accuracy of a
    probe retrained after i = 0..k_max removals, and the C every round used.
    """
    c = float(probe(x, y, groups).C_[0])
    x, x_dev = x.astype(np.float64), x_dev.astype(np.float64)
    basis, accuracy = np.zeros((0, x.shape[1])), []
    for i in range(k_max + 1):
        model = LogisticRegression(C=c, max_iter=5000).fit(x, y)
        accuracy.append(float(model.score(x_dev, y_dev)))
        if i == k_max:
            break
        w = model.coef_[0]
        w = w - basis.T @ (basis @ w)  # already orthogonal up to numerical error
        w /= np.linalg.norm(w)
        basis = np.vstack([basis, w])
        x, x_dev = x - np.outer(x @ w, w), x_dev - np.outer(x_dev @ w, w)
    return basis, accuracy, c


def random_basis(rng: np.random.Generator, dim: int, k_max: int, features: np.ndarray | None = None) -> np.ndarray:
    """k_max orthonormal random directions (rows), nested.

    Isotropic when features is None; otherwise random combinations of the centred
    features, so each direction is drawn in proportion to the variance along it.
    """
    if features is None:
        raw = rng.normal(size=(dim, k_max))
    else:
        centred = features - features.mean(axis=0)
        raw = centred.T @ rng.normal(size=(len(centred), k_max))
    q, _ = np.linalg.qr(raw)
    return q.T

## Sweep

For each direction and seed: learn k_max = max(KS) directions from the source
language's 800 sentences (the seed's Hindi sample, or the first 800 of its
Marathi order), draw RANDOM_REPEATS random subspaces of each kind, then for
every target budget n and every k in KS train the target probe on the first n
target sentences of the seed's order (nested, as in `combine.ipynb`) with k
directions removed and score it on the target dev split. k = 0 is the same
probe for all three conditions. The source probe is also retrained on the
projected source features for each random subspace (at the k in KS), so the
source curves of INLP and the controls can be compared.

Takes a while: that is roughly 2,500 cross-validated probes, split over the
workers.

In [ ]:
def run(direction: str, seed: str):
    source, target = DIRECTIONS[direction]
    rng = np.random.default_rng([int(seed), list(DIRECTIONS).index(direction)])
    k_max = max(KS)
    src_x, src_y, src_g = train[source].rows(samples[source][seed][:SOURCE_SENTENCES])
    dev_x, dev_y = dev[target].x.astype(np.float64), dev[target].y

    inlp_basis, inlp_source, c = inlp(src_x, src_y, src_g, dev[source].x, dev[source].y, k_max)
    conditions = {
        "inlp": [inlp_basis],
        "random": [random_basis(rng, src_x.shape[1], k_max) for _ in range(RANDOM_REPEATS)],
        "random_cov": [random_basis(rng, src_x.shape[1], k_max, src_x) for _ in range(RANDOM_REPEATS)],
    }

    source_runs = [{"direction": direction, "seed": int(seed), "condition": "inlp", "repeat": 0,
                    "k": i, "source_accuracy": acc} for i, acc in enumerate(inlp_source)]
    for name in ["random", "random_cov"]:
        for repeat, basis in enumerate(conditions[name]):
            for k in KS:
                model = LogisticRegression(C=c, max_iter=5000).fit(
                    project(src_x.astype(np.float64), basis[:k]), src_y)
                source_runs.append({"direction": direction, "seed": int(seed), "condition": name,
                                    "repeat": repeat, "k": k,
                                    "source_accuracy": float(model.score(project(dev[source].x.astype(np.float64), basis[:k]),
                                                                         dev[source].y))})

    runs, correct = [], {}
    for n in TARGET_SIZES:
        t_x, t_y, t_g = train[target].rows(samples[target][seed][:n])
        t_x = t_x.astype(np.float64)
        base = probe(t_x, t_y, t_g)
        w0 = base.coef_[0]
        for name, bases in conditions.items():
            for repeat, basis in enumerate(bases):
                for k in KS:
                    b = basis[:k]
                    model = base if k == 0 else probe(project(t_x, b), t_y, t_g)
                    hits = model.predict(project(dev_x, b)) == dev_y
                    # Which dev nouns are right, averaged over a control's random repeats (for the tests below).
                    correct[n, name, k] = correct.get(
                        (n, name, k), 0) + hits.astype(np.float32) / len(bases)
                    runs.append({
                        "direction": direction, "seed": int(seed), "n": n, "condition": name,
                        "repeat": repeat, "k": k, "accuracy": float(hits.mean()),
                        "macro_f1": float(f1_score(dev_y, np.where(hits, dev_y, ~dev_y), average="macro")),
                        # Share of the unprojected target probe's squared weight that lies in the removed subspace.
                        "weight_in_removed": float(np.sum((b @ w0) ** 2) / np.sum(w0 ** 2)),
                        "C": float(model.C_[0]),
                    })
    return runs, source_runs, correct


jobs = [(direction, seed) for direction in DIRECTIONS for seed in SEEDS]
done = Parallel(n_jobs=N_JOBS, verbose=10)(delayed(run)(direction, seed) for direction, seed in jobs)
runs = [row for job_runs, _, _ in done for row in job_runs]
source_runs = [row for _, job_source, _ in done for row in job_source]
# Per dev noun, how much of the time each condition got it right: correct_by_job[direction, seed][n, condition, k].
correct_by_job = {(direction, int(seed)): job_correct for (direction, seed), (_, _, job_correct) in zip(jobs, done)}
print(len(runs), "target probes scored")

## Results

Mean ± standard deviation over the five seeds; within a seed, the three random
subspaces of a control are averaged first. `drop` is the accuracy at k = 0 minus
the accuracy at k, in points, computed seed by seed. `seeds` is the number of
seeds in which removing the source language's directions left the target probe
worse than removing the random ones (isotropic / data covariance), so 5/5 means
INLP hurt more in every seed. `weight in removed` is the share of the target
probe's squared weight (unprojected probe, k = 0) that lies in the removed
subspace.

In [ ]:
CONDITIONS = {"inlp": "Source language's directions (INLP)", "random": "Random, isotropic",
              "random_cov": "Random, data covariance"}
SEEDS_INT = [int(seed) for seed in SEEDS]


def per_seed(rows: list[dict], key: str, direction: str, n: int | None, condition: str) -> np.ndarray:
    """(seeds, len(KS)) array of `key`, with the random repeats of a seed averaged."""
    out = np.zeros((len(SEEDS_INT), len(KS)))
    for s, seed in enumerate(SEEDS_INT):
        for j, k in enumerate(KS):
            out[s, j] = np.mean([r[key] for r in rows if r["direction"] == direction and r["seed"] == seed
                                 and r["condition"] == condition and r["k"] == k and (n is None or r["n"] == n)])
    return out


def stats(a: np.ndarray) -> dict:
    return {"mean": a.mean(axis=0).tolist(), "std": a.std(axis=0).tolist()}


summary, source_summary = {}, {}
for direction in DIRECTIONS:
    summary[direction] = {}
    for n in TARGET_SIZES:
        summary[direction][n] = {}
        accuracy = {c: per_seed(runs, "accuracy", direction, n, c) for c in CONDITIONS}
        for c in CONDITIONS:
            summary[direction][n][c] = {
                "accuracy": stats(accuracy[c]),
                "macro_f1": stats(per_seed(runs, "macro_f1", direction, n, c)),
                "drop": stats(100 * (accuracy[c][:, [0]] - accuracy[c])),
                "weight_in_removed": stats(per_seed(runs, "weight_in_removed", direction, n, c)),
            }
            if c != "inlp":
                summary[direction][n][c]["seeds_inlp_lower"] = (accuracy["inlp"] < accuracy[c]).sum(axis=0).tolist()
    source_summary[direction] = {c: stats(per_seed(source_runs, "source_accuracy", direction, None, c))
                                 for c in CONDITIONS}
# INLP's source curve is kept at every round, not just the k in KS.
inlp_source_curve = {
    direction: {
        "mean": np.mean([[r["source_accuracy"] for r in sorted(
            (r for r in source_runs if r["direction"] == direction and r["seed"] == seed
             and r["condition"] == "inlp"), key=lambda r: r["k"])] for seed in SEEDS_INT], axis=0).tolist(),
        "std": np.std([[r["source_accuracy"] for r in sorted(
            (r for r in source_runs if r["direction"] == direction and r["seed"] == seed
             and r["condition"] == "inlp"), key=lambda r: r["k"])] for seed in SEEDS_INT], axis=0).tolist(),
    } for direction in DIRECTIONS
}

result = {"layer": LAYER, "pooling": POOLING, "split": "dev", "ks": KS, "target_sizes": TARGET_SIZES,
          "source_sentences": SOURCE_SENTENCES, "random_repeats": RANDOM_REPEATS,
          "majority": majority, "summary": summary, "source_summary": source_summary,
          "inlp_source_curve": inlp_source_curve, "runs": runs, "source_runs": source_runs}
(ROOT / "results" / "ablation.json").write_text(json.dumps(result, indent=2) + "\n")

PRINT_KS = [k for k in [0, 1, 2, 4, 8, 16, 32, 64] if k in KS]
columns = [KS.index(k) for k in PRINT_KS]
for direction, (source, target) in DIRECTIONS.items():
    print(f"\n=== Remove {LANGUAGES[source]} directions, probe {LANGUAGES[target]} "
          f"(dev majority {majority[target]:.3f}) ===")
    for n in TARGET_SIZES:
        block = summary[direction][n]
        print(f"\nn = {n}: {LANGUAGES[target]} dev accuracy")
        print(f"  {'k':40s}" + "".join(f"{k:>14d}" for k in PRINT_KS))
        for c, label in CONDITIONS.items():
            print(f"  {label:40s}" + "".join(
                f"{block[c]['accuracy']['mean'][j]:8.3f} ±{block[c]['accuracy']['std'][j]:.3f}" for j in columns))
        print(f"  {'drop, INLP (points)':40s}" + "".join(f"{block['inlp']['drop']['mean'][j]:14.1f}" for j in columns))
        for c in ["random", "random_cov"]:
            print(f"  {'drop, ' + CONDITIONS[c].lower() + ' (points)':40s}"
                  + "".join(f"{block[c]['drop']['mean'][j]:14.1f}" for j in columns))
        for c in ["random", "random_cov"]:
            print(f"  {'seeds INLP lower than ' + CONDITIONS[c].lower():40s}"
                  + "".join(f"{block[c]['seeds_inlp_lower'][j]:>12d}/{len(SEEDS)}" for j in columns))
        print(f"  {'weight in removed, INLP':40s}" + "".join(
            f"{block['inlp']['weight_in_removed']['mean'][j]:14.3f}" for j in columns))
        print(f"  {'weight in removed, random cov':40s}" + "".join(
            f"{block['random_cov']['weight_in_removed']['mean'][j]:14.3f}" for j in columns))
    print(f"\n{LANGUAGES[source]} dev accuracy of the source probe, retrained after removal")
    print(f"  {'k':40s}" + "".join(f"{k:>14d}" for k in PRINT_KS))
    for c, label in CONDITIONS.items():
        print(f"  {label:40s}" + "".join(
            f"{source_summary[direction][c]['mean'][j]:8.3f} ±{source_summary[direction][c]['std'][j]:.3f}"
            for j in columns))

## Plot

Top row: removing Hindi directions and probing Marathi, one panel per target
budget n, plus the Hindi probe's own accuracy on Hindi dev as the directions come
out. Bottom row: the same the other way round. The dotted line is the majority
rate of the dev split being scored; the bands are ± one standard deviation over
the seeds. The x axis is the number of directions removed.

In [ ]:
styles = {"inlp": ("C3", "-"), "random": ("C0", "--"), "random_cov": ("C2", "-.")}


def draw(ax, xs, stat, condition, label=None):
    mean, std = np.array(stat["mean"]), np.array(stat["std"])
    color, style = styles[condition]
    band = 0.5 * np.array(to_rgb(color)) + 0.5 * np.array(to_rgb("grey"))
    ax.fill_between(xs, mean - std, mean + std, color=band, alpha=0.18, linewidth=0)
    ax.plot(xs, mean, color=color, linestyle=style, marker="o", markersize=3, label=label)


ticks = [k for k in [0, 1, 2, 4, 8, 16, 32, 64] if k in KS]
figure, axes = plt.subplots(len(DIRECTIONS), len(TARGET_SIZES) + 1, figsize=(15, 7), sharey="row")
for row, (direction, (source, target)) in zip(axes, DIRECTIONS.items()):
    for ax, n in zip(row[:-1], TARGET_SIZES):
        for c, label in CONDITIONS.items():
            draw(ax, KS, summary[direction][n][c]["accuracy"], c, label)
        ax.axhline(majority[target], color="grey", linestyle=":")
        ax.set(title=f"{LANGUAGES[target]} probe, n = {n}", xlabel=f"{LANGUAGES[source]} directions removed")
    row[0].set_ylabel("Dev accuracy")
    ax = row[-1]
    curve = inlp_source_curve[direction]
    draw(ax, np.arange(len(curve["mean"])), curve, "inlp")
    for c in ["random", "random_cov"]:
        draw(ax, KS, source_summary[direction][c], c)
    ax.axhline(majority[source], color="grey", linestyle=":")
    ax.set(title=f"{LANGUAGES[source]} probe on {LANGUAGES[source]} dev", xlabel=f"{LANGUAGES[source]} directions removed")
    for ax in row:
        ax.set_xscale("symlog", linthresh=1)
        ax.set_xticks(ticks, [str(t) for t in ticks])
        ax.minorticks_off()
axes[0, 0].legend(fontsize=8)
figure.suptitle(f"Layer {LAYER}, {POOLING}-piece", y=1.0)
figure.tight_layout()
figure.savefig(ROOT / "results" / "ablation.png", dpi=150)
plt.show()

## Are the drops bigger than the controls' drops?

The table above says how far accuracy falls; this asks whether INLP's fall is
bigger than a random removal's by more than the luck of which dev sentences we
happened to have. It uses the same tests as `combine.ipynb`, on the same dev
nouns, with the sentence as the unit that gets resampled (nouns in one sentence
are not independent). For each direction, target budget n, control and k > 0 the
table shows:

- **extra drop:** how many points lower INLP's accuracy is than the control's,
  averaged over the seeds (positive: removing the source language's directions
  hurt more than removing random ones);
- **95% interval:** from 10,000 bootstrap resamples of the dev sentences;
- **p:** how often randomly swapping the two conditions' answers, sentence by
  sentence, gives a difference at least this large;
- **Holm p:** p corrected for running all these comparisons at once (every
  direction, n, control and k);
- **seeds lower:** in how many of the five seeds INLP's accuracy was below the
  control's.

As in `combine.ipynb`, the interval and p only cover which dev sentences we had;
how much the result changes with a different training sample shows in the seeds
column and in the ± of the results table. A control's answer for a noun is its
share of right answers over that seed's random subspaces.

In [ ]:
from statsmodels.stats.multitest import multipletests

rows = []
for direction, (source, target) in DIRECTIONS.items():
    sentence = np.unique(dev[target].sentence, return_inverse=True)[1]
    nouns_per_sentence = np.bincount(sentence)
    rng = np.random.default_rng(0)
    draws = rng.integers(0, len(nouns_per_sentence), (10_000, len(nouns_per_sentence)))
    flips = rng.choice(np.array([-1.0, 1.0], dtype=np.float32), draws.shape)
    for n in TARGET_SIZES:
        for control in ["random", "random_cov"]:
            for k in KS[1:]:
                right = lambda condition: [correct_by_job[direction, seed][n, condition, k] for seed in SEEDS_INT]
                # Per sentence: right answers of the control minus those of INLP, averaged over the seeds.
                diff = np.mean([np.bincount(sentence, weights=c - i)
                                for c, i in zip(right(control), right("inlp"))], axis=0)
                observed = diff.sum() / len(sentence)
                boot = diff[draws].sum(1) / nouns_per_sentence[draws].sum(1)
                swapped = (flips * diff).sum(1) / len(sentence)
                rows.append({
                    "direction": direction, "n": n, "control": control, "k": k,
                    "extra_drop": observed, "low": np.percentile(boot, 2.5), "high": np.percentile(boot, 97.5),
                    "p": (np.sum(np.abs(swapped) >= abs(observed)) + 1) / (len(swapped) + 1),
                    "seeds_lower": int(sum(i.mean() < c.mean() for c, i in zip(right(control), right("inlp")))),
                })
for row, holm in zip(rows, multipletests([row["p"] for row in rows], method="holm")[1]):
    row["holm_p"] = holm
(ROOT / "results" / "ablation_significance.json").write_text(json.dumps(rows, indent=2) + "\n")

for direction, (source, target) in DIRECTIONS.items():
    print(f"\n=== Remove {LANGUAGES[source]} directions, probe {LANGUAGES[target]}: INLP vs random ===")
    print(f"{'control':24s}{'n':>5s}{'k':>4s}{'extra drop':>12s}{'95% interval':>18s}{'p':>9s}{'Holm p':>9s}{'seeds lower':>13s}")
    for t in rows:
        if t["direction"] == direction and t["k"] in PRINT_KS:
            print(f"{CONDITIONS[t['control']]:24s}{t['n']:5d}{t['k']:4d}{100 * t['extra_drop']:+11.1f} "
                  f"  [{100 * t['low']:+5.1f}, {100 * t['high']:+5.1f}]{t['p']:9.4f}{t['holm_p']:9.4f}"
                  f"{t['seeds_lower']:9d}/{len(SEEDS)}")